# 01 — Part 1: AOI, 3 × 4 grid, Sentinel-2A seasonal 2019 composite, RGB chips

**Produces**
| Output | Path |
|---|---|
| AOI boundary (EPSG:4326) | `outputs/geojson/aoi_boundary.geojson` |
| 12 grid cells with IDs `r{row}c{col}` | `outputs/geojson/grid_cells.geojson` |
| Scenes used in the composite | `outputs/metadata/s2_scene_inventory_<SEASON_TAG>.csv` |
| Median composite + observation counts (10 m, grid-aligned) | `outputs/rasters/s2_<SEASON_TAG>_median_10m.tif` |
| Valid-data counts per cell | `outputs/tables/cell_valid_data_part1.csv` |
| One RGB chip per cell, 300 DPI | `outputs/chips/rgb/` (display) and `outputs/chips/rgb_native/` (10 m pixels) |
| Overview panel + chip mosaic for the Google Doc | `outputs/figures/` |
| Six cells for visual labelling (seeded) + blank label sheet | `outputs/labels/visual_labels_six_cells.csv` |
| Run metadata | `outputs/metadata/run_metadata_part1.json` |

All settings come from `config.py`; nothing is hard-coded below.
The imagery window is the season set by `SEASON_START_MONTH`–`SEASON_END_MONTH` of `YEAR` (e.g. `SEASON_TAG = 2019m01-m03`); the same window is used for Dynamic World in notebook 02.
Cell IDs: `r1` = northernmost row, `c1` = westernmost column.

## 0. Setup — repo root, imports, requirements log

In [ ]:
# In Colab, clone the repo first and cd into it, e.g.:
# !git clone https://github.com/<user>/lcw-landcover-assessment.git && %cd lcw-landcover-assessment
# %pip install -q -r requirements.txt

import sys, json, datetime as dt
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))          # notebooks/ -> repo root
from src.env_utils import find_repo_root, log_requirements
ROOT = find_repo_root()

import importlib, config as cfg
importlib.reload(cfg)

import numpy as np, pandas as pd, geopandas as gpd, ee
from src import grid as G, gee_utils as GE, chips as CH

# Append any library used here that is not yet pinned in requirements.txt
log_requirements(["earthengine-api", "geopandas", "shapely", "pyproj", "rasterio",
                  "numpy", "pandas", "matplotlib", "pillow"], cfg.REQUIREMENTS_FILE)
print("Repo root:", ROOT)

## 1. Earth Engine authentication
Set the Cloud project once per session, e.g. `%env EE_PROJECT=my-project` (or edit `config.py`).
`ee.Authenticate()` is only triggered when no valid cached credentials exist.

In [ ]:
GE.init_ee(cfg.EE_PROJECT)

## 2. Build the fixed 3 × 4 grid (500 m cells)
Built in the local UTM zone so every cell is exactly 500 m × 500 m, and the NW corner is snapped
to the 10 m pixel lattice — each cell is therefore exactly 50 × 50 Sentinel-2 pixels, with no partial pixels.

In [ ]:
lon, lat = cfg.AOI_CENTER_LONLAT
cells_utm, spec = G.build_grid(lon, lat, cfg.GRID_ROWS, cfg.GRID_COLS,
                               cfg.CELL_SIZE_M, cfg.PIXEL_SIZE_M, cfg.CELL_ID_FMT)

assert len(cells_utm) == 12 and cells_utm.cell_id.is_unique
assert np.allclose(cells_utm.area_m2, cfg.CELL_SIZE_M ** 2)
print(f"AOI: {cfg.AOI_NAME} | CRS {spec['crs']} | {spec['width_px']} x {spec['height_px']} px "
      f"| {cfg.AOI_AREA_KM2:.2f} km² (limit {cfg.MAX_AOI_AREA_KM2})")
cells_utm[["cell_id", "row", "col", "area_m2", "px_row0", "px_row1", "px_col0", "px_col1"]]

## 3. Export boundary and cell IDs as GeoJSON (EPSG:4326, RFC 7946)

In [ ]:
cells_wgs = G.to_wgs84_with_centroids(cells_utm)
aoi_wgs = gpd.GeoDataFrame({"aoi_name": [cfg.AOI_NAME], "area_km2": [cfg.AOI_AREA_KM2]},
                           geometry=[cells_utm.union_all()], crs=cells_utm.crs).to_crs(4326)

geo_meta = {"aoi_name": cfg.AOI_NAME, "aoi_description": cfg.AOI_DESCRIPTION,
            "grid": f"{cfg.GRID_ROWS} rows x {cfg.GRID_COLS} cols, {cfg.CELL_SIZE_M} m cells",
            "cell_id_rule": "r1 = northernmost row, c1 = westernmost column",
            "grid_built_in": spec["crs"], "imagery_window": [cfg.S2_START, cfg.S2_END], "season_tag": cfg.SEASON_TAG}

G.save_geojson(aoi_wgs, cfg.PATHS["geojson"] / "aoi_boundary.geojson", geo_meta)
G.save_geojson(cells_wgs[["cell_id", "row", "col", "area_m2", "centroid_lon", "centroid_lat", "geometry"]],
               cfg.PATHS["geojson"] / "grid_cells.geojson", geo_meta)
cells_wgs[["cell_id", "centroid_lon", "centroid_lat"]]

## 4. Sentinel-2A Level-1C, season of 2019: filter and cloud-screen
Imagery is matched to the Part 2 classifier's training data. EuroSAT was built from **Sentinel-2A** images
that were **not atmospherically corrected**, i.e. Level-1C top-of-atmosphere reflectance. So:
* Collection: `COPERNICUS/S2_HARMONIZED` (Level-1C), platform filter `SPACECRAFT_NAME = Sentinel-2A` (Sentinel-2B excluded).
* Window: `S2_START` (inclusive) to `S2_END` (exclusive), derived from `SEASON_START_MONTH`/`SEASON_END_MONTH`.
* Scene filter: `CLOUDY_PIXEL_PERCENTAGE ≤ 35` — the same limit Dynamic World uses, so both products draw on one scene set.
* Pixel mask: Cloud Score+ `cs_cdf ≥ CS_THRESHOLD`.
* If fewer than `MIN_IMAGES` scenes survive, the notebook warns; it does **not** silently add Sentinel-2B.

Dynamic World is also produced from Level-1C, so the comparison shares the input level.
The scene list is saved so the *same imagery window* can be reused for Dynamic World in Part 2.

In [ ]:
aoi_ee = GE.aoi_rectangle(spec)

s2_used = cfg.S2_COLLECTION
raw, masked = GE.s2_cloud_screened(aoi_ee, cfg.S2_START, cfg.S2_END, s2_used, cfg.MAX_SCENE_CLOUD_PCT,
                                   cfg.CLOUD_SCORE_COLLECTION, cfg.CS_BAND, cfg.CS_THRESHOLD,
                                   platforms=cfg.S2_PLATFORMS)
inventory = GE.image_inventory(raw)
assert set(inventory.spacecraft) <= set(cfg.S2_PLATFORMS), inventory.spacecraft.unique()
if len(inventory) < cfg.MIN_IMAGES:
    print(f"WARNING: only {len(inventory)} {cfg.S2_PLATFORMS} scenes in the window — "
          "consider widening the window or the scene-cloud filter; record this in the report.")
inventory.to_csv(cfg.S2_INVENTORY_CSV, index=False)
print(f"{s2_used}: {len(inventory)} scenes, {inventory.date.min()} → {inventory.date.max()}")
inventory.groupby(inventory.date.str[:7]).size().rename("scenes_per_month").to_frame().T

## 5. Median composite and per-pixel observation counts
`obs_total` = scenes passing the scene filter; `obs_clear` = observations surviving the Cloud Score+ mask.
These are the valid-data counts reported for each cell.

In [ ]:
composite = GE.median_composite(masked, cfg.S2_EXPORT_BANDS)
counts = GE.obs_count_images(raw, masked)
stack_img = composite.addBands(counts)

## 6. Download pixels on the exact grid lattice → GeoTIFF
`computePixels` is called with the grid's own CRS and affine transform, so array pixels map 1:1 onto cell windows.

In [ ]:
arr, band_names = GE.fetch_grid_array(stack_img, spec, cfg.NODATA)
print(arr.shape, band_names)

tif_path = CH.write_geotiff(arr, band_names, spec, cfg.S2_COMPOSITE_TIF, cfg.NODATA)
print("Saved", tif_path.relative_to(ROOT))
bi = {b: i for i, b in enumerate(band_names)}

## 7. Valid-data counts per cell

In [ ]:
rows = []
for _, cell in cells_utm.iterrows():
    win = arr[cell.px_row0:cell.px_row1, cell.px_col0:cell.px_col1]
    rgb_win = win[..., [bi[b] for b in cfg.RGB_BANDS]]
    valid = ~np.any(rgb_win == cfg.NODATA, axis=-1)
    clear = win[..., bi["obs_clear"]]
    rows.append({
        "cell_id": cell.cell_id,
        "n_pixels": valid.size,
        "n_valid_pixels": int(valid.sum()),
        "valid_frac": round(valid.mean(), 4),
        "valid_area_m2": int(valid.sum()) * cfg.PIXEL_SIZE_M ** 2,
        "obs_total_median": float(np.median(win[..., bi["obs_total"]])),
        "obs_clear_median": float(np.median(clear)),
        "obs_clear_min": float(clear.min()),
    })
valid_df = pd.DataFrame(rows)
valid_df.to_csv(cfg.PATHS["tables"] / "cell_valid_data_part1.csv", index=False)
valid_df

## 8. RGB chips, one per cell (PNG, 300 DPI)
* One fixed stretch (`RGB_STRETCH_MIN`–`RGB_STRETCH_MAX`, gamma `RGB_GAMMA`) for all cells — per-cell stretching would make the same surface look different between chips.
* `rgb_native/`: 50 × 50 px, one pixel = 10 m.
* `rgb/`: same pixels, nearest-neighbour ×`CHIP_UPSCALE` for viewing (no new detail is created).

Note: DPI is print metadata only — a 500 m cell holds 50 × 50 Sentinel-2 pixels regardless of DPI.

In [ ]:
rgb_dn = arr[..., [bi[b] for b in cfg.RGB_BANDS]]
rgb8, n_nodata = CH.stretch_rgb(rgb_dn, cfg.RGB_STRETCH_MIN, cfg.RGB_STRETCH_MAX, cfg.RGB_GAMMA, cfg.NODATA)
print("No-data pixels in AOI:", n_nodata)

chip_paths = {}
for _, cell in cells_utm.iterrows():
    chip = rgb8[cell.px_row0:cell.px_row1, cell.px_col0:cell.px_col1]
    CH.save_png(chip, cfg.PATHS["chips_native"] / f"{cell.cell_id}.png", cfg.PNG_DPI, upscale=1)
    chip_paths[cell.cell_id] = CH.save_png(chip, cfg.PATHS["chips_rgb"] / f"{cell.cell_id}.png",
                                           cfg.PNG_DPI, upscale=cfg.CHIP_UPSCALE)
print(len(chip_paths), "chips written")

## 9. Select the six cells for visual labelling (seeded)
Rule: `numpy.random.default_rng(RANDOM_SEED).choice(12 cells, 6, replace=False)` over cells sorted by ID.
The seed is fixed in `config.py` **before** any imagery is inspected, so the selection cannot be steered by what the cells look like.

The label sheet is left blank on purpose: labels, cue and confidence are filled in by hand from the chips, without AI assistance, and before looking at any model or comparator output.

In [ ]:
ids_sorted = sorted(cells_utm.cell_id, key=lambda s: (int(s[1:s.index('c')]), int(s[s.index('c')+1:])))
rng = np.random.default_rng(cfg.RANDOM_SEED)
label_cells = sorted(rng.choice(ids_sorted, cfg.N_VISUAL_LABEL_CELLS, replace=False).tolist(),
                     key=ids_sorted.index)
print("Visual-label cells:", label_cells)

label_sheet = pd.DataFrame({
    "cell_id": label_cells,
    "visual_label": "",          # one of config.CLASSES keys, or 'mixed'
    "secondary_label": "",       # optional, for mixed cells
    "visual_cue": "",            # the one cue that drove the label
    "confidence": "",            # high / medium / low
    "labelled_by": "",
    "labelled_on": "",
    "chip_path": [f"outputs/chips/rgb/{c}.png" for c in label_cells],
})
sheet_path = cfg.PATHS["labels"] / "visual_labels_six_cells.csv"
if sheet_path.exists():
    print("Label sheet already exists — NOT overwritten:", sheet_path.relative_to(ROOT))
else:
    label_sheet.to_csv(sheet_path, index=False)
    print("Blank label sheet written:", sheet_path.relative_to(ROOT))

## 10. Figures for the Google Doc

In [ ]:
title = f"{cfg.AOI_NAME} — Sentinel-2A L1C {cfg.COMPOSITE_METHOD}, season {cfg.SEASON_TAG} ({cfg.S2_START} to {cfg.S2_END} excl.), RGB B4/B3/B2"
CH.overview_panel(rgb8, cells_utm, spec, cfg.PATHS["figures"] / "aoi_grid_overview.png",
                  cfg.PNG_DPI, title=title, highlight=label_cells)
CH.chip_mosaic(chip_paths, cfg.GRID_ROWS, cfg.GRID_COLS,
               cfg.PATHS["figures"] / "chip_mosaic.png", cfg.PNG_DPI, title=title)

from IPython.display import Image as IPImage, display
display(IPImage(cfg.PATHS["figures"] / "aoi_grid_overview.png", width=700))

## 11. Run metadata (everything needed to reproduce and cite this run)

In [ ]:
meta = {
    "run_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    "ee_version": ee.__version__, "ee_project": cfg.EE_PROJECT,
    "aoi": {"name": cfg.AOI_NAME, "description": cfg.AOI_DESCRIPTION,
            "center_lonlat": cfg.AOI_CENTER_LONLAT, "area_km2": cfg.AOI_AREA_KM2},
    "grid": spec,
    "imagery": {"collection_used": s2_used, "processing_level": cfg.S2_PROCESSING_LEVEL,
                "platforms": cfg.S2_PLATFORMS, "season_months": [cfg.SEASON_START_MONTH, cfg.SEASON_END_MONTH], "season_tag": cfg.SEASON_TAG,
                "start": cfg.S2_START, "end_exclusive": cfg.S2_END,
                "composite": cfg.COMPOSITE_METHOD, "n_scenes": len(inventory),
                "scene_filter": f"CLOUDY_PIXEL_PERCENTAGE <= {cfg.MAX_SCENE_CLOUD_PCT}",
                "pixel_mask": f"{cfg.CLOUD_SCORE_COLLECTION}:{cfg.CS_BAND} >= {cfg.CS_THRESHOLD}",
                "bands": cfg.S2_EXPORT_BANDS},
    "rgb": {"bands": cfg.RGB_BANDS, "stretch_dn": [cfg.RGB_STRETCH_MIN, cfg.RGB_STRETCH_MAX],
            "gamma": cfg.RGB_GAMMA, "dpi": cfg.PNG_DPI, "display_upscale": cfg.CHIP_UPSCALE},
    "visual_label_selection": {"rule": "numpy default_rng(seed).choice, no replacement",
                               "seed": cfg.RANDOM_SEED, "cells": label_cells},
    "classes": cfg.CLASSES,
}
(cfg.PATHS["metadata"] / "run_metadata_part1.json").write_text(json.dumps(meta, indent=2, default=str))
print(json.dumps(meta["imagery"], indent=2))

## 12. Output checks

In [ ]:
expected = ([cfg.PATHS["geojson"] / f for f in ["aoi_boundary.geojson", "grid_cells.geojson"]]
            + [cfg.PATHS["chips_rgb"] / f"{i}.png" for i in ids_sorted]
            + [cfg.PATHS["chips_native"] / f"{i}.png" for i in ids_sorted]
            + [tif_path, sheet_path, cfg.PATHS["figures"] / "aoi_grid_overview.png"])
missing = [str(p) for p in expected if not p.exists()]
assert not missing, missing
assert gpd.read_file(cfg.PATHS["geojson"] / "grid_cells.geojson").cell_id.tolist() == ids_sorted
print(f"All {len(expected)} expected outputs present. Part 1 complete.")